# **1. Import Library**

Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV

# **2. Memuat Dataset dari Hasil Clustering**

Memuat dataset hasil clustering dari file CSV ke dalam variabel DataFrame.

In [2]:
df = pd.read_csv("clustering_data.csv")

In [3]:
df = df.select_dtypes(include=[np.number])  # Hanya memilih kolom numerik

In [4]:
# Asumsikan kolom terakhir adalah label
y = df.iloc[:, -1]  # Label
X = df.iloc[:, :-1]  # Fitur

Kolom terakhir pada dataset ini adalah `Cluster` — label hasil segmentasi K-Means pada tahap Clustering sebelumnya (4 kelas: 0, 1, 2, 3, masing-masing merepresentasikan segmen nasabah **Senior Mapan**, **Muda/Pelajar**, **Profesional Mainstream**, dan **Login Attempts Tinggi**). Model klasifikasi pada notebook ini dilatih untuk mempelajari pola yang membedakan keempat segmen tersebut dari fitur transaksinya, sehingga cluster nasabah baru dapat diprediksi tanpa perlu menjalankan ulang proses clustering.

# **3. Data Splitting**

Tahap Data Splitting bertujuan untuk memisahkan dataset menjadi dua bagian: data latih (training set) dan data uji (test set).

In [5]:
# 3. Data Splitting
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
# Normalisasi fitur
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
df.head()

,TransactionAmount,CustomerAge,TransactionDuration,LoginAttempts,AccountBalance,TransactionType_encoded,Channel_encoded,CustomerOccupation_encoded,Cluster
0,14.09,70,81,1,5112.21,1,0,0,2
1,376.24,68,141,1,13758.91,1,0,0,2
2,126.29,19,56,1,1122.35,1,2,3,1
3,184.50,26,25,1,8569.06,1,2,3,1
4,13.45,26,198,1,7429.40,0,2,3,0


# **4. Membangun Model Klasifikasi**


## **a. Membangun Model Klasifikasi**

Setelah memilih algoritma klasifikasi yang sesuai, langkah selanjutnya adalah melatih model menggunakan data latih.

Berikut adalah rekomendasi tahapannya.
1. Pilih algoritma klasifikasi yang sesuai, seperti Logistic Regression, Decision Tree, Random Forest, atau K-Nearest Neighbors (KNN).
2. Latih model menggunakan data latih.

In [7]:
# 4a. Membangun Model Klasifikasi (random forest dan decision tree)
clf_rf = RandomForestClassifier(random_state=42)
clf_rf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [8]:
clf_dt = DecisionTreeClassifier(random_state=42)
clf_dt.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

Dalam eksperimen ini, digunakan dua algoritma klasifikasi: Random Forest dan Decision Tree. 
**1. Decision Tree**
- Decision Tree adalah algoritma berbasis pohon keputusan yang bekerja dengan cara membagi data menjadi beberapa cabang berdasarkan fitur yang paling berpengaruh terhadap target. Algoritma ini membentuk struktur pohon dengan node keputusan (berisi kondisi) dan daun (hasil prediksi).

**2. Random Forest**
- Random Forest adalah pengembangan dari Decision Tree yang menggunakan konsep ensemble learning. Algoritma ini membangun banyak pohon keputusan (tree) dan menggabungkan hasil prediksi dari setiap pohon untuk menghasilkan keputusan akhir berdasarkan voting mayoritas (untuk klasifikasi) atau rata-rata (untuk regresi).

## **b. Evaluasi Model Klasifikasi**

Berikut adalah **rekomendasi** tahapannya.
1. Lakukan prediksi menggunakan data uji.
2. Hitung metrik evaluasi seperti Accuracy dan F1-Score (Opsional: Precision dan Recall).
3. Buat confusion matrix untuk melihat detail prediksi benar dan salah.

In [9]:
y_pred_rf = clf_rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))

print("Random Forest Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))



Random Forest Accuracy: 0.9916666666666667
Random Forest Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.96      0.98        77
           1       0.97      1.00      0.98       118
           2       1.00      1.00      1.00       265
           3       1.00      0.95      0.97        20

    accuracy                           0.99       480
   macro avg       0.99      0.98      0.98       480
weighted avg       0.99      0.99      0.99       480

Random Forest Confusion Matrix:
[[ 74   3   0   0]
 [  0 118   0   0]
 [  0   0 265   0]
 [  0   1   0  19]]


In [10]:
y_pred_dt = clf_dt.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))

print("Decision Tree Classification Report:")
print(classification_report(y_test, y_pred_dt))

print("Decision Tree Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt))

Decision Tree Accuracy: 0.9958333333333333
Decision Tree Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99        77
           1       0.98      1.00      0.99       118
           2       1.00      1.00      1.00       265
           3       1.00      1.00      1.00        20

    accuracy                           1.00       480
   macro avg       1.00      1.00      1.00       480
weighted avg       1.00      1.00      1.00       480

Decision Tree Confusion Matrix:
[[ 76   1   0   0]
 [  0 118   0   0]
 [  0   1 264   0]
 [  0   0   0  20]]


- **Akurasi:** Decision Tree (99,58%) sedikit lebih tinggi dibanding Random Forest (99,17%) pada evaluasi awal (sebelum tuning).

- **Precision & Recall per kelas:**
  - Kedua model sama-sama sempurna (precision & recall 100%) pada Cluster 1 (Muda/Pelajar) dan Cluster 2 (Profesional Mainstream, kelas terbesar dengan 265 data uji).
  - Random Forest sedikit kesulitan di Cluster 0 (Senior Mapan): recall hanya 96% (3 dari 77 sampel salah diklasifikasikan sebagai Cluster 1), dan di Cluster 3 (Login Attempts Tinggi): recall 95% (1 dari 20 sampel salah masuk ke Cluster 1).
  - Decision Tree lebih baik di Cluster 3: recall 100% (seluruh 20 sampel diklasifikasikan benar), namun masih ada 1 sampel Cluster 0 yang salah masuk ke Cluster 1.

- **Confusion Matrix:** Decision Tree memiliki total kesalahan lebih sedikit (2 sampel salah klasifikasi) dibandingkan Random Forest (4 sampel salah klasifikasi), keduanya dari total 480 data uji.

- **Catatan:** Kesalahan klasifikasi pada kedua model konsisten terjadi di batas antara Cluster 0 dan Cluster 1 — masuk akal karena kedua cluster ini sama-sama didominasi channel Branch, sehingga secara fitur transaksi mereka lebih mirip dibanding cluster lainnya.

## **c. Tuning Model Klasifikasi (Optional)**

Gunakan GridSearchCV, RandomizedSearchCV, atau metode lainnya untuk mencari kombinasi hyperparameter terbaik

In [11]:
#RANDOM FOREST
# Definisikan hyperparameter yang akan diuji
param_grid_rf = {
    'n_estimators': [50, 100, 200],  # Jumlah pohon dalam hutan
    'max_depth': [10, 20, None],  # Kedalaman maksimum pohon
    'min_samples_split': [2, 5, 10],  # Minimum sampel untuk membagi node
    'min_samples_leaf': [1, 2, 4]  # Minimum sampel pada setiap leaf node
}

# Inisialisasi model Random Forest
rf = RandomForestClassifier(random_state=42)

# GridSearchCV untuk mencari kombinasi terbaik
grid_search_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search_rf.fit(X_train, y_train)

Fitting 5 folds for each of 81 candidates, totalling 405 fits


GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [10, 20, None],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [50, 100, 200]},
             scoring='accuracy', verbose=1)

In [12]:
# Cetak hasil tuning terbaik
print("Best parameters for Random Forest:", grid_search_rf.best_params_)
best_rf = grid_search_rf.best_estimator_

Best parameters for Random Forest: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}


In [13]:
#DECISION TREE
# Definisikan hyperparameter yang akan diuji
param_grid_dt = {
    'max_depth': [10, 20, None],  # Kedalaman maksimum pohon
    'min_samples_split': [2, 5, 10],  # Minimum sampel untuk membagi node
    'min_samples_leaf': [1, 2, 4]  # Minimum sampel pada setiap leaf node
}

# Inisialisasi model Decision Tree
dt = DecisionTreeClassifier(random_state=42)

# GridSearchCV untuk mencari kombinasi terbaik
grid_search_dt = GridSearchCV(dt, param_grid_dt, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search_dt.fit(X_train, y_train)

Fitting 5 folds for each of 27 candidates, totalling 135 fits


GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [10, 20, None],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10]},
             scoring='accuracy', verbose=1)

In [14]:
# Cetak hasil tuning terbaik
print("Best parameters for Decision Tree:", grid_search_dt.best_params_)
best_dt = grid_search_dt.best_estimator_

Best parameters for Decision Tree: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 10}


## **d. Evaluasi Model Klasifikasi setelah Tuning (Optional)**

Berikut adalah rekomendasi tahapannya.
1. Gunakan model dengan hyperparameter terbaik.
2. Hitung ulang metrik evaluasi untuk melihat apakah ada peningkatan performa.

In [15]:
# Evaluasi Random Forest setelah tuning
y_pred_rf = best_rf.predict(X_test)
print("Random Forest Accuracy after Tuning:", accuracy_score(y_test, y_pred_rf))
print("Random Forest Classification Report after Tuning:\n", classification_report(y_test, y_pred_rf))
print("Random Forest Confusion Matrix after Tuning:\n", confusion_matrix(y_test, y_pred_rf))

Random Forest Accuracy after Tuning: 0.99375
Random Forest Classification Report after Tuning:
               precision    recall  f1-score   support

           0       1.00      0.96      0.98        77
           1       0.98      1.00      0.99       118
           2       1.00      1.00      1.00       265
           3       1.00      1.00      1.00        20

    accuracy                           0.99       480
   macro avg       0.99      0.99      0.99       480
weighted avg       0.99      0.99      0.99       480

Random Forest Confusion Matrix after Tuning:
 [[ 74   3   0   0]
 [  0 118   0   0]
 [  0   0 265   0]
 [  0   0   0  20]]


In [16]:
# Evaluasi Decision Tree setelah tuning
y_pred_dt = best_dt.predict(X_test)
print("Decision Tree Accuracy after Tuning:", accuracy_score(y_test, y_pred_dt))
print("Decision Tree Classification Report after Tuning:\n", classification_report(y_test, y_pred_dt))
print("Decision Tree Confusion Matrix after Tuning:\n", confusion_matrix(y_test, y_pred_dt))


Decision Tree Accuracy after Tuning: 0.9958333333333333
Decision Tree Classification Report after Tuning:
               precision    recall  f1-score   support

           0       1.00      0.99      0.99        77
           1       0.98      1.00      0.99       118
           2       1.00      1.00      1.00       265
           3       1.00      1.00      1.00        20

    accuracy                           1.00       480
   macro avg       1.00      1.00      1.00       480
weighted avg       1.00      1.00      1.00       480

Decision Tree Confusion Matrix after Tuning:
 [[ 76   1   0   0]
 [  0 118   0   0]
 [  0   1 264   0]
 [  0   0   0  20]]


## **e. Analisis Hasil Evaluasi Model Klasifikasi**

**Analisis Perubahan**

- **Random Forest** mengalami **peningkatan** akurasi setelah tuning, dari 99,17% menjadi 99,375%. Peningkatan ini terlihat jelas pada Cluster 3 (Login Attempts Tinggi): recall naik dari 95% menjadi 100% — setelah tuning, seluruh 20 sampel Cluster 3 pada data uji berhasil diklasifikasikan dengan benar (sebelumnya 1 sampel salah masuk ke Cluster 1).

- **Decision Tree** tidak mengalami perubahan akurasi (tetap 99,58%), menunjukkan bahwa hyperparameter tuning tidak memberikan dampak signifikan terhadap model ini — parameter defaultnya sudah cukup optimal untuk data ini.

- Setelah tuning, **Random Forest justru sedikit lebih unggul** dari Decision Tree di kelas minoritas (Cluster 3), meski secara akurasi keseluruhan Decision Tree masih marginal lebih tinggi (99,58% vs 99,375%).

**a. Precision atau Recall Rendah untuk Kelas Tertentu**

*Random Forest (setelah tuning):*
- Recall Cluster 0 (Senior Mapan) masih 96% — 3 dari 77 sampel tetap salah diklasifikasikan sebagai Cluster 1, tidak berubah dari sebelum tuning. Ini konsisten dengan kemiripan fitur transaksi antara Cluster 0 dan Cluster 1 (sama-sama dominan channel Branch).
- Recall Cluster 3 sudah sempurna (100%) setelah tuning.

*Decision Tree (setelah tuning):*
- Masih ada 1 sampel Cluster 2 yang salah masuk ke Cluster 1 (sebelumnya kesalahan ada di Cluster 0→1). Recall tetap tinggi di semua kelas (≥99%).

**b. Overfitting atau Underfitting?**
- Baik Random Forest maupun Decision Tree menunjukkan performa yang sangat tinggi dan konsisten (akurasi >99% pada data uji), tanpa gap besar antara precision dan recall di tiap kelas — tidak ada indikasi underfitting.
- Karena kelas yang diprediksi (`Cluster`) berasal dari label hasil K-Means pada fitur yang sama persis dengan fitur input klasifikasi ini, batas antar kelas memang relatif tegas secara matematis (jarak ke centroid), sehingga akurasi setinggi ini **wajar** dan bukan murni indikasi overfitting seperti pada kasus klasifikasi dengan label independen/asli.
- Cluster 3 sebagai kelas minoritas (hanya 92 dari 2.399 data, atau ~3,8%) tetap terklasifikasi dengan baik setelah tuning — menunjukkan model tidak bias berlebihan ke kelas mayoritas (Cluster 2) meski distribusi kelasnya cukup timpang.
- Untuk pengembangan lebih lanjut, bisa dicoba cross-validation k-fold pada seluruh dataset (bukan hanya 1x train-test split) untuk memastikan stabilitas performa, atau algoritma lain seperti XGBoost/LightGBM sebagai pembanding.